# Camada Bronze

A camada Bronze representa o primeiro estágio da Arquitetura Medalhão. Seu objetivo é armazenar os dados provenientes da origem com o menor número possível de transformações, preservando sua estrutura original.

Nesta etapa, os dados da tabela `ecommerce_pedidos`, provenientes do SQL Server, serão carregados para o Data Lake no formato Delta Lake, sendo persistidos fisicamente no caminho correspondente à camada Bronze.

A utilização do formato Delta Lake permite controle transacional, versionamento dos dados e maior confiabilidade para operações futuras de processamento e atualização.


###Autenticação e Configurações

Este notebook centraliza as configurações de autenticação utilizadas no projeto.

As credenciais são recuperadas de forma segura por meio do Databricks Secret Scope, evitando a exposição de informações sensíveis no código-fonte.

Este notebook é executado por outros notebooks do projeto por meio do comando `%run`, disponibilizando as conexões necessárias com o Azure Data Lake Storage Gen2 e o SQL Server.

In [0]:
%run ./00_autenticação



### Configuração de acesso ao Data Lake pelo Spark

Para realizar a persistência dos dados no Azure Data Lake Storage Gen2 por meio do Spark, é necessário configurar a autenticação OAuth no contexto do Spark.

As credenciais utilizadas são recuperadas do Databricks Secret Scope e não são armazenadas diretamente no notebook.

In [0]:
storage_account = "internshipdatalake"
container_name = "squad1"

spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
df_bronze = spark.read.jdbc(
    url=jdbc_url,
    table="ecommerce_pedidos",
    properties=connection_properties
)

In [0]:
df_pedidos_delta = df_bronze.withColumn(
    "data_ultima_atualizacao",
    df_bronze["dt_ultima_atualizacao_status"].cast("date")
)

display(df_pedidos_delta)

In [0]:
df_pedidos_delta.select(
    "data_ultima_atualizacao"
).distinct().orderBy(
    "data_ultima_atualizacao"
).show()

In [0]:
df_pedidos_pandas = df_pedidos_delta.toPandas()

print(df_pedidos_pandas.shape)
print(df_pedidos_pandas.dtypes)

In [0]:
storage_account = "internshipdatalake"
container_name = "squad1"

bronze_path = (
    f"abfss://{container_name}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"bronze/ecommerce_pedidos"
)

print(bronze_path)

In [0]:
print(storage_account)
print(bool(client_id))
print(bool(tenant_id))
print(bool(client_secret))

In [0]:
df_bronze = spark.read.jdbc(
    url=jdbc_url,
    table="ecommerce_pedidos",
    properties=connection_properties
)

In [0]:
from pyspark.sql.functions import col

df_pedidos_delta = df_bronze.withColumn(
    "data_ultima_atualizacao",
    col("dt_ultima_atualizacao_status").cast("date")
)

print(df_pedidos_delta.count())

In [0]:
df_pedidos_delta.select(
    "data_ultima_atualizacao"
).distinct().orderBy(
    "data_ultima_atualizacao"
).show()

In [0]:
df_pedidos_pandas = df_pedidos_delta.toPandas()

print(df_pedidos_pandas.shape)
print(df_pedidos_pandas.dtypes)

In [0]:
import pandas as pd

df_pedidos_pandas["data_ultima_atualizacao"] = (
    pd.to_datetime(
        df_pedidos_pandas["data_ultima_atualizacao"]
    ).dt.date
)

print(df_pedidos_pandas["data_ultima_atualizacao"].dtype)

In [0]:
print(df_pedidos_pandas["data_ultima_atualizacao"].unique())

In [0]:
from deltalake import write_deltalake

write_deltalake(
    bronze_path,
    df_pedidos_pandas,
    mode="append",
    partition_by=["data_ultima_atualizacao"],
    storage_options={
        "AZURE_STORAGE_ACCOUNT_NAME": storage_account,
        "AZURE_STORAGE_CLIENT_ID": client_id,
        "AZURE_STORAGE_TENANT_ID": tenant_id,
        "AZURE_STORAGE_CLIENT_SECRET": client_secret
    }
)

print("Carga Bronze concluída!")

In [0]:
fs_squad1 = service_client.get_file_system_client(
    file_system="squad1"
)

for item in fs_squad1.get_paths("bronze/ecommerce_pedidos"):
    print(item.name)

In [0]:
from deltalake import DeltaTable

dt_pedidos = DeltaTable(
    bronze_path,
    storage_options={
        "AZURE_STORAGE_ACCOUNT_NAME": storage_account,
        "AZURE_STORAGE_CLIENT_ID": client_id,
        "AZURE_STORAGE_TENANT_ID": tenant_id,
        "AZURE_STORAGE_CLIENT_SECRET": client_secret
    }
)

print("Versão do Delta:", dt_pedidos.version())

In [0]:
df_delta = dt_pedidos.to_pandas()

print(df_delta.shape)

In [0]:
df_bronze.groupBy("id_pedido") \
    .count() \
    .filter("count > 1") \
    .show()

In [0]:
df_bronze.filter(
    col("id_pedido") == 1769126164
).orderBy(
    "dt_ultima_atualizacao_status"
).show(truncate=False)

In [0]:
from pyspark.sql.functions import count, countDistinct

df_bronze.groupBy("id_pedido") \
    .agg(
        count("*").alias("quantidade"),
        countDistinct("dt_ultima_atualizacao_status").alias("datas_atualizacao")
    ) \
    .filter("quantidade > 1") \
    .show(truncate=False)

In [0]:
df_bronze.filter(
    col("id_pedido") == 1771277160
).orderBy(
    "dt_ultima_atualizacao_status"
).show(truncate=False)

In [0]:
from pyspark.sql.functions import countDistinct, struct

df_bronze.groupBy("id_pedido") \
    .agg(
        countDistinct(
            struct(
                "id_cliente",
                "id_endereco_entrega",
                "dt_pedido",
                "status_pedido",
                "valor_total",
                "valor_frete",
                "metodo_pagamento",
                "dt_ultima_atualizacao_status"
            )
        ).alias("versoes_diferentes")
    ) \
    .filter("versoes_diferentes > 1") \
    .show(truncate=False)

In [0]:
id_atualizados = [
    1771277160,
    1771651441,
    1771651440,
    1771621320,
    1771584840,
    1770333000,
    1770331800
]

df_bronze.filter(
    col("id_pedido").isin(id_atualizados)
).select(
    "id_pedido",
    "status_pedido",
    "valor_total",
    "valor_frete",
    "metodo_pagamento",
    "dt_ultima_atualizacao_status"
).orderBy(
    "id_pedido",
    "dt_ultima_atualizacao_status"
).show(truncate=False)

In [0]:
df_bronze.select(
    "id_pedido"
).distinct().count()

In [0]:
from pyspark.sql.functions import count

df_bronze.groupBy("id_pedido") \
    .agg(
        count("*").alias("quantidade")
    ) \
    .filter("quantidade > 1") \
    .count()

In [0]:
from pyspark.sql.functions import count, countDistinct, struct

df_classificacao = df_bronze.groupBy("id_pedido") \
    .agg(
        count("*").alias("quantidade"),
        countDistinct(
            struct(
                "id_cliente",
                "id_endereco_entrega",
                "dt_pedido",
                "status_pedido",
                "valor_total",
                "valor_frete",
                "metodo_pagamento",
                "dt_ultima_atualizacao_status"
            )
        ).alias("versoes_diferentes")
    ) \
    .filter("quantidade > 1") \
    .withColumn(
        "tipo_ocorrencia",
        when(
            col("versoes_diferentes") == 1,
            "DUPLICADO"
        ).otherwise("ATUALIZACAO")
    )

df_classificacao.show(truncate=False)

In [0]:
from pyspark.sql.functions import sum, when

df_resumo = df_bronze.groupBy("id_pedido") \
    .agg(
        count("*").alias("quantidade")
    )

df_resumo.select(
    sum(
        when(col("quantidade") == 1, 1).otherwise(0)
    ).alias("pedidos_unicos"),
    sum(
        when(col("quantidade") > 1, col("quantidade") - 1).otherwise(0)
    ).alias("ocorrencias_adicionais")
).show()

In [0]:
from pyspark.sql.functions import current_timestamp

df_controle = df_classificacao.withColumn(
    "data_ingestao",
    current_timestamp()
)

df_controle.show(truncate=False)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_pedido = Window.partitionBy(
    "id_pedido"
).orderBy(
    col("dt_ultima_atualizacao_status").desc()
)

df_ultima_versao = df_bronze.withColumn(
    "rn",
    row_number().over(window_pedido)
).filter(
    col("rn") == 1
).drop("rn")

print("Pedidos únicos na última versão:", df_ultima_versao.count())

In [0]:
df_bronze.groupBy(
    "id_pedido",
    "dt_ultima_atualizacao_status"
).count() \
.filter("count > 1") \
.show(truncate=False)

In [0]:
from pyspark.sql.functions import current_timestamp

df_estado_atual = df_ultima_versao.select(
    "id_pedido",
    "dt_ultima_atualizacao_status"
).withColumn(
    "data_ingestao",
    current_timestamp()
)

df_estado_atual.show(10, truncate=False)

In [0]:
print("Registros no estado atual:", df_estado_atual.count())

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_pedido = Window.partitionBy(
    "id_pedido"
).orderBy(
    col("dt_ultima_atualizacao_status")
)

df_ocorrencias = df_bronze.withColumn(
    "numero_ocorrencia",
    row_number().over(window_pedido)
)

df_ocorrencias.select(
    "id_pedido",
    "numero_ocorrencia",
    "dt_ultima_atualizacao_status"
).show(20, truncate=False)

In [0]:
from pyspark.sql.functions import sha2, concat_ws, coalesce, lit

df_ocorrencias = df_bronze.withColumn(
    "hash_registro",
    sha2(
        concat_ws(
            "||",
            coalesce(col("id_pedido").cast("string"), lit("")),
            coalesce(col("id_cliente").cast("string"), lit("")),
            coalesce(col("id_endereco_entrega").cast("string"), lit("")),
            coalesce(col("dt_pedido").cast("string"), lit("")),
            coalesce(col("status_pedido"), lit("")),
            coalesce(col("valor_total").cast("string"), lit("")),
            coalesce(col("valor_frete").cast("string"), lit("")),
            coalesce(col("metodo_pagamento"), lit("")),
            coalesce(col("dt_ultima_atualizacao_status").cast("string"), lit(""))
        ),
        256
    )
)

In [0]:
df_ocorrencias.filter(
    col("id_pedido") == 1769126160
).select(
    "id_pedido",
    "dt_ultima_atualizacao_status",
    "hash_registro"
).show(truncate=False)

In [0]:
df_ocorrencias.filter(
    col("id_pedido") == 1771277160
).select(
    "id_pedido",
    "dt_ultima_atualizacao_status",
    "status_pedido",
    "valor_total",
    "hash_registro"
).show(truncate=False)

In [0]:
from pyspark.sql.functions import countDistinct, when

df_classificacao = (
    df_ocorrencias
    .groupBy("id_pedido")
    .agg(
        count("*").alias("quantidade"),
        countDistinct("hash_registro").alias("versoes_diferentes")
    )
    .filter(col("quantidade") > 1)
    .withColumn(
        "tipo_ocorrencia",
        when(
            col("versoes_diferentes") == 1,
            "DUPLICADO"
        ).otherwise(
            "ATUALIZACAO"
        )
    )
)

In [0]:
df_classificacao.orderBy("id_pedido").show(
    50,
    truncate=False
)

In [0]:
df_controle = df_ocorrencias.join(
    df_classificacao.select(
        "id_pedido",
        "tipo_ocorrencia"
    ),
    on="id_pedido",
    how="left"
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_pedido = Window.partitionBy(
    "id_pedido"
).orderBy(
    col("dt_ultima_atualizacao_status")
)

df_ocorrencias = df_ocorrencias.withColumn(
    "numero_ocorrencia",
    row_number().over(window_pedido)
)

In [0]:
df_controle = df_ocorrencias.join(
    df_classificacao.select(
        "id_pedido",
        "tipo_ocorrencia"
    ),
    on="id_pedido",
    how="left"
)

In [0]:
df_controle.select(
    "id_pedido",
    "numero_ocorrencia",
    "dt_ultima_atualizacao_status",
    "hash_registro",
    "tipo_ocorrencia"
).orderBy(
    "id_pedido",
    "numero_ocorrencia"
).show(30, truncate=False)

In [0]:
from pyspark.sql.functions import coalesce, lit

df_controle_primeira_carga = df_controle.withColumn(
    "tipo_ocorrencia",
    coalesce(
        col("tipo_ocorrencia"),
        lit("NOVO")
    )
)

In [0]:
df_controle_primeira_carga.groupBy(
    "tipo_ocorrencia"
).count().show()

In [0]:
from pyspark.sql.functions import current_timestamp

df_ingestion_control = df_controle_primeira_carga.select(
    "id_pedido",
    "hash_registro",
    "dt_ultima_atualizacao_status",
    "tipo_ocorrencia"
).withColumn(
    "data_ingestao",
    current_timestamp()
)

In [0]:
df_ingestion_control.printSchema()

In [0]:
df_ingestion_control.show(20, truncate=False)

In [0]:
df_ingestion_control_pandas = df_ingestion_control.toPandas()

print(df_ingestion_control_pandas.shape)
print(df_ingestion_control_pandas.dtypes)

In [0]:
control_path = (
    f"abfss://squad1@"
    f"{storage_account}.dfs.core.windows.net/"
    f"metadata/ingestion_control_log"
)

print(control_path)

In [0]:
from deltalake import write_deltalake

write_deltalake(
    control_path,
    df_ingestion_control_pandas,
    mode="append",
    storage_options={
        "AZURE_STORAGE_ACCOUNT_NAME": storage_account,
        "AZURE_STORAGE_CLIENT_ID": client_id,
        "AZURE_STORAGE_TENANT_ID": tenant_id,
        "AZURE_STORAGE_CLIENT_SECRET": client_secret
    }
)

print("Ingestion Control Log gravado com sucesso!")

In [0]:
from deltalake import DeltaTable

dt_control = DeltaTable(
    control_path,
    storage_options={
        "AZURE_STORAGE_ACCOUNT_NAME": storage_account,
        "AZURE_STORAGE_CLIENT_ID": client_id,
        "AZURE_STORAGE_TENANT_ID": tenant_id,
        "AZURE_STORAGE_CLIENT_SECRET": client_secret
    }
)

print("Versão do Delta:", dt_control.version())

In [0]:
df_control_delta = dt_control.to_pandas()

print("Quantidade de registros:", len(df_control_delta))
print("Quantidade de colunas:", len(df_control_delta.columns))

In [0]:
df_historico_controle = dt_control.to_pandas()

print(df_historico_controle.shape)
df_historico_controle.head()

In [0]:
df_nova_carga = spark.read.jdbc(
    url=jdbc_url,
    table="ecommerce_pedidos",
    properties=connection_properties
)

print("Quantidade de registros:", df_nova_carga.count())

In [0]:
from pyspark.sql.functions import sha2, concat_ws, coalesce, lit, col

df_nova_carga = df_nova_carga.withColumn(
    "hash_registro",
    sha2(
        concat_ws(
            "||",
            coalesce(col("id_pedido").cast("string"), lit("")),
            coalesce(col("id_cliente").cast("string"), lit("")),
            coalesce(col("id_endereco_entrega").cast("string"), lit("")),
            coalesce(col("dt_pedido").cast("string"), lit("")),
            coalesce(col("status_pedido"), lit("")),
            coalesce(col("valor_total").cast("string"), lit("")),
            coalesce(col("valor_frete").cast("string"), lit("")),
            coalesce(col("metodo_pagamento"), lit("")),
            coalesce(col("dt_ultima_atualizacao_status").cast("string"), lit(""))
        ),
        256
    )
)

In [0]:
df_nova_carga.select(
    "id_pedido",
    "hash_registro"
).show(10, truncate=False)

In [0]:
df_historico_controle[
    df_historico_controle["id_pedido"] == 1769126160
]

In [0]:
df_historico_chaves = df_historico_controle[
    ["id_pedido", "hash_registro"]
].drop_duplicates()

print("Registros únicos no histórico:", len(df_historico_chaves))

In [0]:
df_nova_carga_chaves = df_nova_carga.select(
    "id_pedido",
    "hash_registro"
).dropDuplicates()

print("Registros únicos na nova carga:", df_nova_carga_chaves.count())

In [0]:
df_historico_chaves_spark = spark.createDataFrame(
    df_historico_chaves
)

df_historico_chaves_spark.printSchema

In [0]:
df_comparacao = df_nova_carga_chaves.join(
    df_historico_chaves_spark,
    on=["id_pedido", "hash_registro"],
    how="left"
)

df_comparacao.show(10)

In [0]:
df_historico_chaves_spark = df_historico_chaves_spark.withColumn(
    "encontrado_historico",
    lit(1)
)

In [0]:
df_comparacao = df_nova_carga_chaves.join(
    df_historico_chaves_spark,
    on=["id_pedido", "hash_registro"],
    how="left"
)

In [0]:
df_comparacao.show(10)

In [0]:
df_historico_ids = df_historico_controle[
    ["id_pedido"]
].drop_duplicates()

print("IDs únicos no histórico:", len(df_historico_ids))

In [0]:
df_historico_ids_spark = spark.createDataFrame(
    df_historico_ids
)

df_historico_ids_spark.show(5)

In [0]:
df_classificacao_nova = df_nova_carga.join(
    df_historico_ids_spark.withColumn(
        "id_existente_historico",
        lit(1)
    ),
    on="id_pedido",
    how="left"
)

In [0]:
df_classificacao_nova.select(
    "id_pedido",
    "hash_registro",
    "id_existente_historico"
).show(10, truncate=False)

In [0]:
df_classificacao_nova = df_classificacao_nova.join(
    df_comparacao.select(
        "id_pedido",
        "hash_registro",
        "encontrado_historico"
    ),
    on=["id_pedido", "hash_registro"],
    how="left"
)

In [0]:
df_classificacao_nova.select(
    "id_pedido",
    "hash_registro",
    "id_existente_historico",
    "encontrado_historico"
).show(10, truncate=False)

In [0]:
from pyspark.sql.functions import when, lit

df_classificacao_nova = df_classificacao_nova.withColumn(
    "tipo_ocorrencia",
    when(
        col("id_existente_historico").isNull(),
        "NOVO"
    ).when(
        col("encontrado_historico").isNotNull(),
        "DUPLICADO"
    ).otherwise(
        "ATUALIZACAO"
    )
)

In [0]:
df_classificacao_nova.groupBy(
    "tipo_ocorrencia"
).count().show()

In [0]:
df_nova_carga.filter(
    col("id_pedido") == 1771277160
).select(
    "id_pedido",
    "status_pedido",
    "valor_total",
    "valor_frete",
    "dt_ultima_atualizacao_status",
    "hash_registro"
).show(truncate=False)

In [0]:
df_teste_atualizacao = df_nova_carga.filter(
    col("id_pedido") == 1771277160
).limit(1)

In [0]:
df_teste_atualizacao = df_teste_atualizacao.withColumn(
    "valor_total",
    lit(200.00)
)

In [0]:
df_teste_atualizacao.select(
    "id_pedido",
    "valor_total",
    "hash_registro"
).show(truncate=False)

In [0]:
df_teste_atualizacao = df_teste_atualizacao.withColumn(
    "hash_registro",
    sha2(
        concat_ws(
            "||",
            coalesce(col("id_pedido").cast("string"), lit("")),
            coalesce(col("id_cliente").cast("string"), lit("")),
            coalesce(col("id_endereco_entrega").cast("string"), lit("")),
            coalesce(col("dt_pedido").cast("string"), lit("")),
            coalesce(col("status_pedido"), lit("")),
            coalesce(col("valor_total").cast("string"), lit("")),
            coalesce(col("valor_frete").cast("string"), lit("")),
            coalesce(col("metodo_pagamento"), lit("")),
            coalesce(col("dt_ultima_atualizacao_status").cast("string"), lit(""))
        ),
        256
    )
)

In [0]:
df_teste_atualizacao.select(
    "id_pedido",
    "valor_total",
    "hash_registro"
).show(truncate=False)

In [0]:
df_teste_atualizacao.join(
    df_historico_chaves_spark,
    on=["id_pedido", "hash_registro"],
    how="left"
).select(
    "id_pedido",
    "valor_total",
    "hash_registro"
).show(truncate=False)

In [0]:
df_teste_atualizacao = df_teste_atualizacao.join(
    df_historico_ids_spark.withColumn(
        "id_existente_historico",
        lit(1)
    ),
    on="id_pedido",
    how="left"
)

In [0]:
df_teste_atualizacao = df_teste_atualizacao.withColumn(
    "tipo_ocorrencia",
    when(
        col("id_existente_historico").isNull(),
        "NOVO"
    ).otherwise(
        "ATUALIZACAO"
    )
)

In [0]:
df_teste_atualizacao.select(
    "id_pedido",
    "valor_total",
    "hash_registro",
    "id_existente_historico",
    "tipo_ocorrencia"
).show(truncate=False)

In [0]:
id_novo = df_nova_carga.selectExpr(
    "max(id_pedido) + 1 as id_novo"
).collect()[0]["id_novo"]

print(id_novo)

In [0]:
df_teste_novo = df_nova_carga.limit(1).withColumn(
    "id_pedido",
    lit(id_novo)
)

In [0]:
print(id_novo)

In [0]:
df_historico_ids_spark.filter(
    col("id_pedido") == 1777331644
).show()

In [0]:
df_teste_novo = df_teste_novo.join(
    df_historico_ids_spark.withColumn(
        "id_existente_historico",
        lit(1)
    ),
    on="id_pedido",
    how="left"
)

In [0]:
df_teste_novo = df_teste_novo.withColumn(
    "tipo_ocorrencia",
    when(
        col("id_existente_historico").isNull(),
        "NOVO"
    ).otherwise(
        "ATUALIZACAO"
    )
)

In [0]:
df_teste_novo.select(
    "id_pedido",
    "valor_total",
    "hash_registro",
    "id_existente_historico",
    "tipo_ocorrencia"
).show(truncate=False)

In [0]:
df_teste_novo = df_nova_carga.limit(1).withColumn(
    "id_pedido",
    lit(1777331644)
)

In [0]:
df_teste_novo.printSchema()

In [0]:
df_teste_novo = df_teste_novo.join(
    df_historico_ids_spark.withColumn(
        "id_existente_historico",
        lit(1)
    ),
    on="id_pedido",
    how="left"
)

In [0]:
df_teste_novo.select(
    "id_pedido",
    "id_existente_historico"
).show

In [0]:
df_teste_novo.select(
    "id_pedido",
    "id_existente_historico"
).show()

In [0]:
df_teste_novo = df_teste_novo.withColumn(
    "tipo_ocorrencia",
    when(
        col("id_existente_historico").isNull(),
        "NOVO"
    ).otherwise(
        "ATUALIZACAO"
    )
)

In [0]:
df_teste_novo.select(
    "id_pedido",
    "id_existente_historico",
    "tipo_ocorrencia"
).show()

In [0]:
df_teste_duplicado = df_nova_carga.filter(
    col("id_pedido") == 1769126160
).limit(1)

In [0]:
df_teste_duplicado.select(
    "id_pedido",
    "valor_total",
    "hash_registro"
).show(truncate=False)

In [0]:
df_carga_teste = (
    df_teste_novo
    .unionByName(df_teste_atualizacao, allowMissingColumns=True)
    .unionByName(df_teste_duplicado, allowMissingColumns=True)
)

In [0]:
df_carga_teste.select(
    "id_pedido",
    "valor_total",
    "hash_registro"
).show(truncate=False)

In [0]:
df_ids_historicos = df_historico_ids_spark.withColumn(
    "id_existente_historico",
    lit(1)
)


In [0]:
df_carga_teste_classificacao = df_carga_teste.join(
    df_ids_historicos,
    on="id_pedido",
    how="left"
)

In [0]:
df_teste_novo = df_nova_carga.limit(1).withColumn(
    "id_pedido",
    lit(1777331644)
)

In [0]:
df_teste_atualizacao = df_nova_carga.filter(
    col("id_pedido") == 1771277160
).limit(1).withColumn(
    "valor_total",
    lit(200.00)
)

In [0]:
df_teste_duplicado = df_nova_carga.filter(
    col("id_pedido") == 1769126160
).limit(1)

In [0]:
df_teste_novo.printSchema()
df_teste_atualizacao.printSchema()
df_teste_duplicado.printSchema()

In [0]:
df_teste_novo = df_teste_novo.withColumn(
    "id_pedido",
    col("id_pedido").cast("long")
)

In [0]:
df_teste_atualizacao = df_teste_atualizacao.withColumn(
    "hash_registro",
    sha2(
        concat_ws(
            "||",
            coalesce(col("id_pedido").cast("string"), lit("")),
            coalesce(col("id_cliente").cast("string"), lit("")),
            coalesce(col("id_endereco_entrega").cast("string"), lit("")),
            coalesce(col("dt_pedido").cast("string"), lit("")),
            coalesce(col("status_pedido"), lit("")),
            coalesce(col("valor_total").cast("string"), lit("")),
            coalesce(col("valor_frete").cast("string"), lit("")),
            coalesce(col("metodo_pagamento"), lit("")),
            coalesce(col("dt_ultima_atualizacao_status").cast("string"), lit(""))
        ),
        256
    )
)

In [0]:
df_carga_teste = (
    df_teste_novo
    .unionByName(df_teste_atualizacao)
    .unionByName(df_teste_duplicado)
)

In [0]:
df_carga_teste.select(
    "id_pedido",
    "valor_total",
    "hash_registro"
).show(truncate=False)

In [0]:
df_chaves_historicas = df_historico_chaves_spark.select(
    "id_pedido",
    "hash_registro"
).dropDuplicates().withColumn(
    "encontrado_historico",
    lit(1)
)

In [0]:
df_carga_teste_classificacao = df_carga_teste.join(
    df_chaves_historicas,
    on=["id_pedido", "hash_registro"],
    how="left"
)

In [0]:
df_carga_teste_classificacao.select(
    "id_pedido",
    "hash_registro",
    "encontrado_historico"
).show(truncate=False)

In [0]:
df_carga_teste_classificacao = df_carga_teste_classificacao.join(
    df_ids_historicos,
    on="id_pedido",
    how="left"
)

In [0]:
df_carga_teste_classificacao.select(
    "id_pedido",
    "encontrado_historico",
    "id_existente_historico"
).show()

In [0]:
df_carga_teste_classificacao = df_carga_teste_classificacao.withColumn(
    "tipo_ocorrencia",
    when(
        col("id_existente_historico").isNull(),
        "NOVO"
    ).when(
        col("encontrado_historico").isNotNull(),
        "DUPLICADO"
    ).otherwise(
        "ATUALIZACAO"
    )
)

In [0]:
df_carga_teste_classificacao.select(
    "id_pedido",
    "valor_total",
    "tipo_ocorrencia"
).show()

In [0]:
df_para_bronze = df_carga_teste_classificacao.filter(
    col("tipo_ocorrencia").isin(
        "NOVO",
        "ATUALIZACAO"
    )
)

In [0]:
df_para_bronze.select(
    "id_pedido",
    "valor_total",
    "tipo_ocorrencia"
).show()

In [0]:
df_para_bronze = df_para_bronze.withColumn(
    "data_ultima_atualizacao",
    col("dt_ultima_atualizacao_status").cast("date")
)

In [0]:
df_para_bronze.select(
    "id_pedido",
    "valor_total",
    "dt_ultima_atualizacao_status",
    "data_ultima_atualizacao",
    "tipo_ocorrencia"
).show(truncate=False)

In [0]:
df_bronze_final = df_para_bronze.drop(
    "tipo_ocorrencia"
)

In [0]:
df_bronze_final.printSchema()

In [0]:
storage_account = "internshipdatalake"
container_name = "squad1"

bronze_path = (
    f"abfss://{container_name}@"
    f"{storage_account}.dfs.core.windows.net/"
    f"bronze/ecommerce_pedidos"
)

print(bronze_path)

### Leitura dos dados de origem

Nesta etapa, os dados da tabela `ecommerce_pedidos`, armazenados no SQL Server, são carregados para um DataFrame Spark.

A leitura é realizada por meio da conexão JDBC configurada no notebook de autenticação.

In [0]:
df_bronze = spark.read.jdbc(
    url=jdbc_url,
    table="ecommerce_pedidos",
    properties=connection_properties
)

### Persistência da camada Bronze

Nesta etapa, os dados provenientes do SQL Server serão persistidos no Azure Data Lake Storage Gen2, na camada Bronze, utilizando o formato Delta Lake.

A persistência será realizada no caminho físico `squad1/bronze/ecommerce_pedidos`.

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("append") \
    .save(bronze_path)

In [0]:
.save(bronze_path)

###Estrutura da Arquitetura Medalhão

Para organizar os dados de acordo com a arquitetura medalhão, serão criados três schemas no Unity Catalog:

Bronze: armazenamento dos dados brutos provenientes da origem.
Silver: armazenamento dos dados tratados e padronizados.
Gold: armazenamento dos dados agregados e preparados para consumo analítico.

Essa separação facilita a governança, rastreabilidade e manutenção dos pipelines de dados.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

#Criação dos Schemas para as 3 camadas, para facilitar a organização.

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

#Validação do schemas das camadas criados.

### Adição de metadados de carga

Nesta etapa é adicionada a coluna data_carga ao DataFrame da camada Bronze.

Essa coluna registra o momento em que os dados foram ingeridos na plataforma, permitindo rastrear quando cada carga foi realizada e facilitando auditorias, monitoramento e análises históricas do processo de ingestão.

A inclusão de metadados de carga é uma prática comum em arquiteturas de dados modernas, contribuindo para a governança e rastreabilidade das informações.

In [0]:
#Primeiro vamos adicionar uma coluna técnica de carga.

from pyspark.sql.functions import current_timestamp

df_bronze = (
    df_pedidos
    .withColumn("data_carga", current_timestamp())
)



In [0]:
df_bronze.printSchema()

#Conferindo se a tabela está com a coluna data_carga.

### Persistência da camada Bronze

Após a inclusão dos metadados de carga, os dados serão persistidos na camada Bronze utilizando o formato Delta Lake.

A gravação será realizada no modo append, permitindo que novas cargas sejam adicionadas à tabela sem sobrescrever os dados já existentes.

O uso do Delta Lake oferece recursos como transações ACID, versionamento, controle de schema e melhor desempenho para processamento analítico.

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list("mercadata-adls-oauth")

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.ecommerce_pedidos")

### Validação da camada Bronze

Após a persistência dos dados na camada Bronze, foi realizada uma validação para confirmar a criação da tabela Delta e verificar a disponibilidade dos registros armazenados.

Essa etapa tem como objetivo garantir que os dados provenientes do SQL Server foram carregados corretamente para o Unity Catalog, preservando sua estrutura original e os metadados de ingestão adicionados durante o processo.

A validação consiste na consulta dos registros armazenados na tabela workspace.bronze.ecommerce_pedidos, permitindo verificar a integridade dos dados e confirmar o sucesso da carga realizada na camada Bronze

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM workspace.bronze.ecommerce_pedidos
""").show()

In [0]:
display(
    spark.table("workspace.bronze.ecommerce_pedidos")
)

In [0]:
spark.sql("""
DESCRIBE DETAIL workspace.bronze.ecommerce_pedidos
""").select("location").show(truncate=False)

In [0]:
spark.sql("""
DESCRIBE TABLE EXTENDED workspace.bronze.ecommerce_pedidos
""").filter(
    "col_name IN ('Type', 'Provider', 'Location')"
).show(truncate=False)

In [0]:
spark.sql("""
DESCRIBE SCHEMA EXTENDED workspace.bronze
""").show(truncate=False)